In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.optimize import curve_fit
from matplotlib.patches import Rectangle
from signals.PeakSignal import PeakSignal
from skimage.transform import rotate, radon
from tools import analyse_waveforms, save_analysis, read_analysis, H5FileManager

In [ ]:
sample_name = "test"
wafer_type = "test"
data_folder = "/path/to/yourT7/sample"

In [ ]:
import h5py
import numpy as np
from tqdm import tqdm


def check_150v_for_nans(filename):
    target_v = "voltage_0150V"
    print(f"--- Fast Scan: {filename} ({target_v}) ---")

    with h5py.File(filename, "r") as f:
        if target_v not in f:
            print(f" Group {target_v} not found in the file.")
            return

        v_grp = f[target_v]
        positions = [
            name for name in v_grp.keys() if isinstance(v_grp[name], h5py.Group)
        ]

        corrupted_positions = []

        for pos_name in tqdm(positions, desc=f"Scanning {target_v}", unit="pos"):
            pos_grp = v_grp[pos_name]

            for ds_name in pos_grp.keys():
                obj = pos_grp[ds_name]

                if isinstance(obj, h5py.Dataset):
                    data = obj[:]

                    if np.isnan(data).any():
                        nan_rows = np.where(np.isnan(data).any(axis=1))[0]
                        tqdm.write(
                            f"⚠️ NaN in {pos_name} -> {ds_name} | Triggers: {nan_rows.tolist()}"
                        )
                        corrupted_positions.append(pos_name)

        print("\n" + "=" * 30)
        if not corrupted_positions:
            print(f" Status: Group {target_v} is clean.")
        else:
            unique_bad = len(set(corrupted_positions))
            print(f" Alert: Found {unique_bad} corrupted positions in {target_v}.")

In [ ]:
# filename = f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}.h5"
# check_150v_for_nans(filename)

In [ ]:
# voltages = []

In [ ]:
# for voltage in voltages:
#     df = analyse_waveforms(
#         f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}.h5", voltage
#     )
#     save_analysis(
#         df, f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}_{voltage}V.csv"
#     )

In [ ]:
# save_analysis(df, f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}_{voltage}V.csv")

In [ ]:
# H5FileManager.split_by_voltage(f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}.h5")
# H5FileManager.merge_files(f"{data_folder}/{wafer_type}/{sample_name}/combined_results.h5", files_to_combine)

In [ ]:
def sigmoid(x, amp, cen, wid):
    return amp / (1 + np.exp(-1 * (x - cen) / wid))


def sigmoid_inverse(x, amp, cen, wid):
    return cen - wid * np.log(amp / x - 1)


def gauss(x, amp, cen, wid):
    return amp * np.exp(-((x - cen) ** 2) / 2 / wid / wid)


def sigmoid_inverse_error(x, amp, cen, wid, d_amp, d_cen, d_wid):
    return np.sqrt(
        d_cen**2
        + np.log(amp / x - 1) * np.log(amp / x - 1) * d_wid**2
        + wid**2 / (amp / x - 1) / (amp / x - 1) * d_amp**2 / x / x
    )

In [ ]:
def plot_amplitude_map(df, voltage, x_borders=None, y_borders=None):
    agg_df = (
        df[df["pulse_number"] == 1]
        .groupby(["x", "y"], as_index=False)["peak_amplitude"]
        .mean()
    )
    pivot = agg_df.pivot(index="y", columns="x", values="peak_amplitude")
    pivot = pivot.sort_index().sort_index(axis=1)

    _, ax = plt.subplots(figsize=(10, 8))

    sns.heatmap(
        np.abs(pivot),
        cmap="plasma",
        cbar_kws={"label": "Mean amplitude [V]"},
        ax=ax,
    )

    ax.set_title(f"TCT Scan - {voltage} V - Combined Channels")
    ax.set_xlabel(r"x ($\mu$m)")
    ax.set_ylabel(r"y ($\mu$m)")

    if x_borders is not None and y_borders is not None:

        x_vals = pivot.columns.values
        y_vals = pivot.index.values

        for key in x_borders.keys():

            xb = x_borders[key]
            yb = y_borders[key]

            x0 = np.searchsorted(x_vals, xb[0])
            x1 = np.searchsorted(x_vals, xb[1])
            y0 = np.searchsorted(y_vals, yb[0])
            y1 = np.searchsorted(y_vals, yb[1])

            rect = Rectangle(
                (x0, y0),
                x1 - x0,
                y1 - y0,
                linewidth=2,
                edgecolor="black",
                facecolor="none",
            )

            ax.add_patch(rect)

    plt.tight_layout()
    plt.show()

In [ ]:
voltage = 150

In [ ]:
df = read_analysis(
    f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}_{voltage}V.csv"
)

In [ ]:
plot_amplitude_map(df, voltage)

In [ ]:
def get_corrected_df(df):
    pivot_for_angle = (
        df[df["pulse_number"] == 1]
        .pivot_table(index="y", columns="x", values="peak_amplitude", aggfunc="mean")
        .fillna(0)
    )
    image = np.abs(pivot_for_angle.values)
    theta = np.linspace(0.0, 180.0, 180, endpoint=False)
    sinogram = radon(image, theta=theta, circle=False)
    best_angle_deg = theta[np.argmax(np.var(sinogram, axis=0))]

    x_orig_coords = np.sort(df["x"].unique())
    y_orig_coords = np.sort(df["y"].unique())
    dx = np.median(np.diff(x_orig_coords))
    dy = np.median(np.diff(y_orig_coords))
    x_mid, y_mid = x_orig_coords.mean(), y_orig_coords.mean()

    prefixes = ("peak_time", "peak_amplitude", "peak_integral")
    cols_to_rotate = [c for c in df.columns if c.startswith(prefixes)]

    corrected_chunks = []

    for ch in df["ch"].unique():
        df_ch = df[(df["ch"] == ch) & (df["pulse_number"] == 1)]
        if df_ch.empty:
            continue

        rotated_data = {}
        h_new, w_new = 0, 0

        for col in cols_to_rotate:
            if col not in df_ch.columns:
                continue

            pivot_col = df_ch.pivot_table(
                index="y", columns="x", values=col, aggfunc="mean"
            ).fillna(0)

            rotated_img = rotate(
                pivot_col.values,
                -best_angle_deg,
                resize=True,
                order=3,
                mode="constant",
                cval=0,
            )
            rotated_data[col] = rotated_img.flatten()
            h_new, w_new = rotated_img.shape

        xi = np.linspace(x_mid - (w_new * dx) / 2, x_mid + (w_new * dx) / 2, w_new)
        yi = np.linspace(y_mid - (h_new * dy) / 2, y_mid + (h_new * dy) / 2, h_new)
        xi_grid, yi_grid = np.meshgrid(xi, yi)

        chunk = pd.DataFrame(
            {
                "x": xi_grid.flatten(),
                "y": yi_grid.flatten(),
                "ch": ch,
                "pulse_number": 1,
            }
        )

        for col, values in rotated_data.items():
            chunk[col] = values

        corrected_chunks.append(chunk)

    df_final = pd.concat(corrected_chunks, ignore_index=True)
    df_final["x"] = df_final["x"].round(1)
    df_final["y"] = df_final["y"].round(1)

    print(f"Sensor aligned! Angle: {-best_angle_deg:.2f}°")
    return df_final

In [ ]:
df_corrected = get_corrected_df(df)

In [ ]:
plot_amplitude_map(df_corrected, voltage)

In [ ]:
def plot_y_profile(df, voltage, threshold=0.9, make_plot=False):
    y_borders = defaultdict(list)
    fit_popt_pcov = {}

    if make_plot:
        plt.figure(figsize=(10, 6))

    for ch in sorted(df["ch"].unique()):
        df_ch = df[(df["ch"] == ch) & (df["pulse_number"] == 1)]

        agg_df = df_ch.groupby("y", as_index=False)["peak_amplitude"].sum()
        ylabel = "Sum of mean amplitudes [V]"

        agg_df = agg_df.sort_values("y")

        y_arr = agg_df["y"].values
        amp_arr = np.abs(agg_df["peak_amplitude"].values)

        if make_plot:
            plt.plot(
                y_arr,
                amp_arr,
                marker="o",
                label=f"Channel {ch}",
            )

        max_amp = np.max(amp_arr)
        plateau_mask = amp_arr > (0.5 * max_amp)
        pad_center = np.median(y_arr[plateau_mask])

        left_mask = y_arr <= pad_center
        right_mask = y_arr > pad_center

        x_left = y_arr[left_mask]
        y_left = amp_arr[left_mask]

        x_right = y_arr[right_mask]
        y_right = amp_arr[right_mask]

        amp_guess_l = np.max(y_left)
        x0_guess_l = x_left[np.argmin(np.abs(y_left - (0.5 * amp_guess_l)))]

        popt_left, pcov_left = curve_fit(
            sigmoid,
            x_left,
            y_left,
            p0=[amp_guess_l, x0_guess_l, 1],
            maxfev=10000,
        )

        amp_guess_r = np.max(y_right)
        x0_guess_r = x_right[np.argmin(np.abs(y_right - (0.5 * amp_guess_r)))]

        popt_right, pcov_right = curve_fit(
            sigmoid,
            x_right,
            y_right,
            p0=[amp_guess_r, x0_guess_r, -1],
            maxfev=10000,
        )

        border_left = sigmoid_inverse(threshold * popt_left[0], *popt_left)
        border_right = sigmoid_inverse(threshold * popt_right[0], *popt_right)

        y_borders[f"ch_{ch}"].extend([border_left, border_right])

        fit_popt_pcov[f"ch_{ch}"] = {
            "left": (popt_left, pcov_left),
            "right": (popt_right, pcov_right),
        }

        if make_plot:
            plt.axvline(border_left, color="black", linestyle="--")
            plt.axvline(border_right, color="black", linestyle="--")

    min_key = min(y_borders, key=lambda k: min(y_borders[k]))
    max_key = max(y_borders, key=lambda k: max(y_borders[k]))

    inner_edge_bottom = max(y_borders[min_key])
    inner_edge_top = min(y_borders[max_key])

    popt_bottom, pcov_bottom = fit_popt_pcov[min_key]["right"]
    popt_top, pcov_top = fit_popt_pcov[max_key]["left"]

    perr_bottom = np.sqrt(np.diag(pcov_bottom))
    perr_top = np.sqrt(np.diag(pcov_top))

    amp_b, cen_b, wid_b = popt_bottom
    d_amp_b, d_cen_b, d_wid_b = perr_bottom
    x_target_b = threshold * amp_b

    amp_t, cen_t, wid_t = popt_top
    d_amp_t, d_cen_t, d_wid_t = perr_top
    x_target_t = threshold * amp_t

    err_bottom_edge = sigmoid_inverse_error(
        x_target_b, amp_b, cen_b, wid_b, d_amp_b, d_cen_b, d_wid_b
    )
    err_top_edge = sigmoid_inverse_error(
        x_target_t, amp_t, cen_t, wid_t, d_amp_t, d_cen_t, d_wid_t
    )

    interpad_distance = inner_edge_top - inner_edge_bottom
    interpad_distance_error = np.sqrt(err_bottom_edge**2 + err_top_edge**2)

    y_borders["interpad"] = [inner_edge_top, inner_edge_bottom]

    if make_plot:
        plt.title(f"TCT Scan - {voltage} V - Y Profile")
        plt.xlabel(r"y ($\mu$m)")
        plt.ylabel(ylabel)
        plt.text(
            0.5,
            0.15,
            rf"Inter-pad distance: {interpad_distance:.1f} $\pm$ {interpad_distance_error:.1f} $\mu$m, {threshold*100:.0f}% threshold",
            ha="center",
            va="center",
            transform=plt.gca().transAxes,
            fontsize=12,
            bbox=dict(facecolor="white", alpha=0.8),
        )
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    return dict(y_borders), interpad_distance, interpad_distance_error

In [ ]:
y_borders, interpad_distance, interpad_distance_error = plot_y_profile(
    df_corrected, voltage, threshold=0.9, make_plot=True
)

In [ ]:
def plot_x_profile(df, voltage, threshold=0.9, make_plot=False):
    x_borders = defaultdict(list)

    if make_plot:
        plt.figure(figsize=(8, 6))

    for ch in sorted(df["ch"].unique()):

        df_ch = df[(df["ch"] == ch) & (df["pulse_number"] == 1)]
        agg_df = df_ch.groupby("x", as_index=False)["peak_amplitude"].sum()
        xlabel = "Sum of mean amplitudes [V]"

        agg_df = agg_df.sort_values("x")
        agg_df = agg_df[agg_df["x"] != 0]

        if make_plot:
            plt.plot(
                agg_df["x"],
                np.abs(agg_df["peak_amplitude"]),
                marker="o",
                label=f"Channel {ch}",
            )

        peak_idx = np.argmax(np.abs(agg_df["peak_amplitude"].values))
        peak_x = agg_df["x"].values[peak_idx]

        left_mask = agg_df["x"] <= peak_x
        right_mask = agg_df["x"] >= peak_x

        x_left = agg_df["x"][left_mask].values
        y_left = np.abs(agg_df["peak_amplitude"][left_mask].values)

        x_right = agg_df["x"][right_mask].values
        y_right = np.abs(agg_df["peak_amplitude"][right_mask].values)

        popt_left, _ = curve_fit(
            sigmoid,
            x_left,
            y_left,
            p0=[np.max(y_left), np.median(x_left), 1],
            maxfev=10000,
        )
        x_borders[f"ch_{ch}"].append(
            sigmoid_inverse(threshold * popt_left[0], *popt_left)
        )

        popt_right, _ = curve_fit(
            sigmoid,
            x_right,
            y_right,
            p0=[np.max(y_right), np.median(x_right), -1],
            maxfev=10000,
        )
        x_borders[f"ch_{ch}"].append(
            sigmoid_inverse(threshold * popt_right[0], *popt_right)
        )

    all_lefts = [b[0] for k, b in x_borders.items() if k.startswith("ch_")]
    all_rights = [b[1] for k, b in x_borders.items() if k.startswith("ch_")]

    if all_lefts and all_rights:
        x_borders["interpad"] = [np.mean(all_lefts), np.mean(all_rights)]

    if make_plot:
        channels = [
            int(k.split("_")[1]) for k in x_borders.keys() if k.startswith("ch_")
        ]
        if channels:
            min_ch_num = min(channels)

            for ch in sorted(df["ch"].unique()):
                borders = x_borders.get(f"ch_{ch}")
                if borders and len(borders) == 2:
                    color = "tab:blue" if ch == min_ch_num else "tab:orange"
                    plt.axvline(min(borders), color=color, linestyle="--")
                    plt.axvline(max(borders), color=color, linestyle="--")

        plt.title(f"TCT Scan - {voltage} V - X Profile")
        plt.xlabel(r"x ($\mu$m)")
        plt.ylabel(xlabel)
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    return dict(x_borders)

In [ ]:
x_borders = plot_x_profile(df_corrected, voltage, threshold=0.7, make_plot=True)

In [ ]:
plot_amplitude_map(df_corrected, voltage, x_borders=x_borders, y_borders=y_borders)

In [ ]:
def plot_jitter(
    df,
    percentage,
    x_borders=None,
    y_borders=None,
    fit_options={
        "bins": 100,
        "left_lim": 98.3,
        "right_lim": 99.0,
        "peak_left": 98.45,
        "peak_right": 98.9,
    },
    make_plot=False,
):
    bins = fit_options["bins"]
    left_lim, right_lim = fit_options["left_lim"], fit_options["right_lim"]
    peak_left, peak_right = fit_options["peak_left"], fit_options["peak_right"]

    min_std = (peak_right - peak_left) * 0.01

    if make_plot:
        plt.figure(figsize=(10, 6))

    jitters = defaultdict(list)
    jitter_errors = defaultdict(list)

    for ch in sorted(df["ch"].unique()):
        df_ch = df[df["ch"] == ch]

        if x_borders and f"ch_{ch}" in x_borders and x_borders[f"ch_{ch}"] is not None:
            df_ch = df_ch[
                (df_ch["x"] >= min(x_borders[f"ch_{ch}"]))
                & (df_ch["x"] <= max(x_borders[f"ch_{ch}"]))
            ]
        if y_borders and f"ch_{ch}" in y_borders and y_borders[f"ch_{ch}"] is not None:
            df_ch = df_ch[
                (df_ch["y"] >= min(y_borders[f"ch_{ch}"]))
                & (df_ch["y"] <= max(y_borders[f"ch_{ch}"]))
            ]

        pivot = df_ch.pivot_table(
            index=["x", "y", "z", "waveform_index", "ch"],
            columns="pulse_number",
            values=f"peak_time_{percentage}",
        )

        pivot = pivot.dropna(subset=[1, 2])
        jitter = pivot[2] - pivot[1]

        if len(jitter) == 0:
            print(f"Skipping Ch {ch}: No data left after filtering.")
            continue

        y, bin_edges = np.histogram(
            jitter.values,
            bins=int(bins / (right_lim - left_lim) * (peak_right - peak_left)),
            range=(peak_left, peak_right),
        )
        x = (bin_edges[:-1] + bin_edges[1:]) / 2

        if np.sum(y) == 0:
            print(
                f"Skipping curve fit for Ch {ch}: No data falls within the peak bounds."
            )
            continue

        window_mask = (jitter.values >= peak_left) & (jitter.values <= peak_right)
        window_data = jitter.values[window_mask]

        safe_amp = np.max(y)
        safe_mean = np.clip(x[np.argmax(y)], peak_left + 1e-5, peak_right - 1e-5)
        safe_std = (
            max(np.std(window_data), min_std) if len(window_data) > 0 else min_std
        )

        fit_success = False
        try:
            popt, pcov = curve_fit(
                gauss,
                x,
                y,
                p0=[safe_amp, safe_mean, safe_std],
                maxfev=10000,
                bounds=(
                    [0, peak_left, 0],
                    [np.inf, peak_right, np.inf],
                ),
            )
            perr = np.sqrt(np.diag(pcov))
            std_dev = popt[2]
            std_dev_err = perr[2]
            jitters[f"ch_{ch}"] = std_dev
            jitter_errors[f"ch_{ch}"] = std_dev_err
            fit_success = True
            hist_label = rf"Jitter Ch {ch} ($\sigma$={1e3*std_dev:.3f}$\pm${1e3*std_dev_err:.3f} ps)"
        except Exception as e:
            print(f"Warning: Curve fit failed for Ch {ch}. Reason: {e}")
            hist_label = rf"Jitter Ch {ch} (Fit Failed)"

        if make_plot:
            plt.hist(
                jitter.values,
                bins=bins,
                range=(left_lim, right_lim),
                alpha=0.5,
                label=hist_label,
                zorder=1,
            )
            if fit_success:
                x_plot = np.linspace(x[0], x[-1], 1000)
                plt.plot(x_plot, gauss(x_plot, *popt), "k-", linewidth=2)

    df_interpad = df.copy()

    if x_borders and "interpad" in x_borders and x_borders["interpad"] is not None:
        df_interpad = df_interpad[
            (df_interpad["x"] >= min(x_borders["interpad"]))
            & (df_interpad["x"] <= max(x_borders["interpad"]))
        ]

    if y_borders and "interpad" in y_borders and y_borders["interpad"] is not None:
        df_interpad = df_interpad[
            (df_interpad["y"] >= min(y_borders["interpad"]))
            & (df_interpad["y"] <= max(y_borders["interpad"]))
        ]

    pivot = df_interpad.pivot_table(
        index=["x", "y", "z", "waveform_index", "ch"],
        columns="pulse_number",
        values=f"peak_time_{percentage}",
    )

    pivot = pivot.dropna(subset=[1, 2])
    jitter = pivot[2] - pivot[1]

    if len(jitter) > 0:
        y, bin_edges = np.histogram(
            jitter.values,
            bins=int(bins / (right_lim - left_lim) * (peak_right - peak_left)),
            range=(peak_left, peak_right),
        )
        x = (bin_edges[:-1] + bin_edges[1:]) / 2

        if np.sum(y) > 0:
            window_mask = (jitter.values >= peak_left) & (jitter.values <= peak_right)
            window_data = jitter.values[window_mask]

            safe_amp_interpad = np.max(y)
            safe_mean_interpad = np.clip(
                x[np.argmax(y)], peak_left + 1e-5, peak_right - 1e-5
            )
            safe_std_interpad = (
                max(np.std(window_data), min_std) if len(window_data) > 0 else min_std
            )

            fit_success = False
            try:
                popt, pcov = curve_fit(
                    gauss,
                    x,
                    y,
                    p0=[safe_amp_interpad, safe_mean_interpad, safe_std_interpad],
                    maxfev=10000,
                    bounds=(
                        [0, peak_left, 0],
                        [np.inf, peak_right, np.inf],
                    ),
                )
                perr = np.sqrt(np.diag(pcov))
                std_dev = popt[2]
                std_dev_err = perr[2]
                jitters["interpad"] = std_dev
                jitter_errors["interpad"] = std_dev_err
                fit_success = True
                hist_label = rf"Jitter inter-pad ($\sigma$={1e3*std_dev:.3f}$\pm${1e3*std_dev_err:.3f} ps)"
            except Exception as e:
                print(f"Warning: Curve fit failed for interpad. Reason: {e}")
                hist_label = r"Jitter inter-pad (Fit Failed)"

            if make_plot:
                plt.hist(
                    jitter.values,
                    bins=bins,
                    range=(left_lim, right_lim),
                    alpha=0.5,
                    label=hist_label,
                    zorder=2,
                )
                if fit_success:
                    x_plot = np.linspace(x[0], x[-1], 1000)
                    plt.plot(x_plot, gauss(x_plot, *popt), "k-", linewidth=2)
        else:
            print("Skipping fit for interpad: Histogram is empty in the peak range.")
    else:
        print("Skipping interpad: No data left after filtering.")

    df_full = df.copy()

    ch_y_mins = []
    ch_y_maxs = []
    if y_borders is not None:
        for ch in df["ch"].unique():
            key = f"ch_{ch}"
            if key in y_borders and y_borders[key] is not None:
                ch_y_mins.append(min(y_borders[key]))
                ch_y_maxs.append(max(y_borders[key]))

    if ch_y_mins and ch_y_maxs:
        y_min_full = min(ch_y_mins)
        y_max_full = max(ch_y_maxs)
        df_full = df_full[(df_full["y"] >= y_min_full) & (df_full["y"] <= y_max_full)]

    if (
        x_borders is not None
        and "interpad" in x_borders
        and x_borders["interpad"] is not None
    ):
        x_min_full = min(x_borders["interpad"])
        x_max_full = max(x_borders["interpad"])
        df_full = df_full[(df_full["x"] >= x_min_full) & (df_full["x"] <= x_max_full)]

    pivot_full = df_full.pivot_table(
        index=["x", "y", "z", "waveform_index", "ch"],
        columns="pulse_number",
        values=f"peak_time_{percentage}",
    )

    pivot_full = pivot_full.dropna(subset=[1, 2])
    jitter_full = pivot_full[2] - pivot_full[1]
    all_jitter_values = jitter_full.values

    if len(all_jitter_values) > 0:
        y, bin_edges = np.histogram(
            all_jitter_values,
            bins=int(bins / (right_lim - left_lim) * (peak_right - peak_left)),
            range=(peak_left, peak_right),
        )
        x = (bin_edges[:-1] + bin_edges[1:]) / 2

        if np.sum(y) > 0:
            window_mask = (all_jitter_values >= peak_left) & (
                all_jitter_values <= peak_right
            )
            window_data = all_jitter_values[window_mask]

            safe_amp_full = np.max(y)
            safe_mean_full = np.clip(
                x[np.argmax(y)], peak_left + 1e-5, peak_right - 1e-5
            )
            safe_std_full = (
                max(np.std(window_data), min_std) if len(window_data) > 0 else min_std
            )

            fit_success = False
            try:
                popt_full, pcov_full = curve_fit(
                    gauss,
                    x,
                    y,
                    p0=[safe_amp_full, safe_mean_full, safe_std_full],
                    maxfev=10000,
                    bounds=(
                        [0, peak_left, 0],
                        [np.inf, peak_right, np.inf],
                    ),
                )
                perr_full = np.sqrt(np.diag(pcov_full))
                std_dev_full = popt_full[2]
                std_dev_err_full = perr_full[2]
                jitters["full"] = std_dev_full
                jitter_errors["full"] = std_dev_err_full
                fit_success = True
                hist_label = rf"Jitter full ($\sigma$={1e3*std_dev_full:.3f}$\pm${1e3*std_dev_err_full:.3f} ps)"
            except Exception as e:
                print(f"Warning: Curve fit failed for full dataset. Reason: {e}")
                hist_label = r"Jitter full (Fit Failed)"

            if make_plot:
                plt.hist(
                    all_jitter_values,
                    bins=bins,
                    range=(left_lim, right_lim),
                    alpha=0.5,
                    label=hist_label,
                    zorder=0,
                )
                if fit_success:
                    x_plot = np.linspace(x[0], x[-1], 1000)
                    plt.plot(x_plot, gauss(x_plot, *popt_full), "k-", linewidth=2)
        else:
            print(
                "Skipping fit for full dataset: Histogram is empty in the peak range."
            )
    else:
        print(
            "Skipping full dataset: No valid jitter values accumulated within the custom bounds."
        )

    if make_plot:
        plt.xlabel("Pulses delay, [ns]")
        plt.legend()

    return dict(jitters), dict(jitter_errors)

In [ ]:
percentage = 20
jitters, jitter_errors = plot_jitter(
    df,
    percentage,
    x_borders=x_borders,
    y_borders=y_borders,
    fit_options={
        "bins": 100,
        "left_lim": 98,
        "right_lim": 100,
        "peak_left": 98,
        "peak_right": 100,
    },
    make_plot=True,
)

In [ ]:
def plot_waveform(sample_path, voltage, position, channel, index=0):
    target_pos = np.array(position)

    with h5py.File(sample_path, "r") as f:
        grp_name = f"voltage_{voltage:04d}V"
        if grp_name not in f:
            print(f"Error: Group {grp_name} not found in HDF5 file.")
            return

        voltage_grp = f[grp_name]
        closest_key = None
        closest_pos = None
        min_dist = float("inf")

        for key in voltage_grp.keys():
            subgroup = voltage_grp[key]

            if all(k in subgroup.attrs for k in ("x", "y", "z")):
                curr_x = 1e6 * subgroup.attrs["x"]
                curr_y = 1e6 * subgroup.attrs["y"]
                curr_z = 1e6 * subgroup.attrs["z"]

                current_pos = np.array([curr_x, curr_y, curr_z])

                dist = np.linalg.norm(current_pos - target_pos)

                if dist < min_dist:
                    min_dist = dist
                    closest_key = key
                    closest_pos = current_pos

        if closest_key is None:
            print("Could not find any groups with x, y, z attributes.")
            return

        if min_dist == 0:
            print(f"Found exact match: {closest_pos} in group '{closest_key}'")
        else:
            print(f"Position {position} not found.")
            print(
                f"Closest match: {closest_pos} in group '{closest_key}' (Dist: {min_dist:.2f})"
            )

        pos_grp = voltage_grp[closest_key]

        ds_name = f"ch{channel}_v"

        ch_data = pos_grp[ds_name]

        dt = ch_data.attrs["dt"]
        t0 = ch_data.attrs["t0"]

        n_samples = ch_data.shape[1]
        half = n_samples // 2

        t = t0 + np.arange(n_samples) * dt
        t *= 1e9

        waveforms = ch_data[:][index]
        plt.plot(t[:half], waveforms[:half], label="Pulse 1")
        plt.plot(t[half:], waveforms[half:], label="Pulse 2")
        plt.xlabel("t, [ns]")
        plt.ylabel("Amplitude, [V]")
        plt.title(
            f"{voltage} V, ch={channel} x={position[0]} y={position[1]} z={position[2]}, wf_idx={index}"
        )
        plt.legend()

In [ ]:
position, ch = [-2440, -1850, 68000], 1
plot_waveform(
    f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}.h5",
    voltage,
    position,
    ch,
    0,
)

In [ ]:
def plot_mean_charge_map(df, voltage, x_borders, y_borders, make_plot=False):

    mean_per_ch = (
        df[df["pulse_number"] == 1]
        .groupby(["x", "y", "ch"], as_index=False)["peak_integral"]
        .mean()
    )

    agg_df = mean_per_ch.groupby(["x", "y"], as_index=False)["peak_integral"].sum()

    mean_charges = {}

    if make_plot:
        pivot = agg_df.pivot(index="y", columns="x", values="peak_integral")
        pivot = pivot.sort_index().sort_index(axis=1)

        _, ax = plt.subplots(figsize=(10, 8))

        sns.heatmap(
            np.abs(pivot),
            cmap="plasma",
            cbar_kws={"label": r"Total sum of impedance-charge [V$\cdot$ns]"},
            ax=ax,
        )

        ax.set_title(f"TCT Scan - {voltage} V - Combined Channels")
        ax.set_xlabel(r"x ($\mu$m)")
        ax.set_ylabel(r"y ($\mu$m)")

        x_vals = pivot.columns.values
        y_vals = pivot.index.values

    for key in x_borders.keys():
        if key not in y_borders:
            continue

        xb = x_borders[key]
        yb = y_borders[key]

        if xb is None or yb is None:
            continue

        mask = (
            (agg_df["x"] >= min(xb))
            & (agg_df["x"] <= max(xb))
            & (agg_df["y"] >= min(yb))
            & (agg_df["y"] <= max(yb))
        )
        region_mean = np.abs(agg_df[mask]["peak_integral"]).mean()
        mean_charges[key] = region_mean

        if make_plot:
            x0 = np.searchsorted(x_vals, xb[0])
            x1 = np.searchsorted(x_vals, xb[1])
            y0 = np.searchsorted(y_vals, yb[0])
            y1 = np.searchsorted(y_vals, yb[1])

            rect = Rectangle(
                (x0, y0),
                x1 - x0,
                y1 - y0,
                linewidth=2,
                edgecolor="black",
                facecolor="none",
            )
            ax.add_patch(rect)

    ch_y_mins = []
    ch_y_maxs = []
    for k, v in y_borders.items():
        if k.startswith("ch_") and v is not None:
            ch_y_mins.append(min(v))
            ch_y_maxs.append(max(v))

    if (
        ch_y_mins
        and ch_y_maxs
        and "interpad" in x_borders
        and x_borders["interpad"] is not None
    ):
        y_min_full = min(ch_y_mins)
        y_max_full = max(ch_y_maxs)
        x_min_full = min(x_borders["interpad"])
        x_max_full = max(x_borders["interpad"])

        mask_full = (
            (agg_df["x"] >= x_min_full)
            & (agg_df["x"] <= x_max_full)
            & (agg_df["y"] >= y_min_full)
            & (agg_df["y"] <= y_max_full)
        )
        mean_charges["full"] = np.abs(agg_df[mask_full]["peak_integral"]).mean()

    if make_plot:
        plt.tight_layout()
        plt.show()

    return mean_charges

In [ ]:
mean_charges = plot_mean_charge_map(
    df_corrected, voltage, x_borders=x_borders, y_borders=y_borders, make_plot=True
)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

plt.figure(figsize=(10, 6))

sample_list = [
    ["W11", "TW5_V1", []],
    ["W11", "TW5_V2", []],
    ["W11", "TW1_V2", []],
    ["W11", "TW2_V2", []],
]

for wafer_type, sample_name, exclusions in sample_list:
    folder_name = f"{data_folder}/{wafer_type}/{sample_name}"
    x = []
    y = []
    y_err = []

    for file in tqdm(os.listdir(folder_name)):
        if file.endswith(".csv"):
            with open(f"{folder_name}/{file}", "r") as f:
                df = pd.read_csv(f)

            voltage = int(file.split("_")[-1][:-5])
            if voltage in exclusions:
                continue

            _, interpad_distance, interpad_distance_error = plot_y_profile(
                df, voltage, threshold=0.9
            )

            x.append(voltage)
            y.append(interpad_distance)
            y_err.append(interpad_distance_error)

    if len(x) > 0:
        x_sorted, y_sorted, y_err_sorted = zip(*sorted(zip(x, y, y_err)))

        plt.errorbar(
            x_sorted,
            y_sorted,
            yerr=y_err_sorted,
            marker="o",
            capsize=4,
            label=f"{wafer_type} {sample_name}",
        )

plt.xlabel("Bias voltage, [V]")
plt.grid(True)
plt.ylabel(r"Inter-pad distance, [$\mu$m]")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

sample_list = [
    ["W11", "TW5_V1", []],
    ["W11", "TW5_V2", []],
    ["W11", "TW1_V2", []],
    ["W11", "TW2_V2", []],
]

for wafer_type, sample_name, exclusions in sample_list:
    folder_name = f"{data_folder}/{wafer_type}/{sample_name}"

    x = []
    y_ch1, y_ch1_err = [], []
    y_ch2, y_ch2_err = [], []
    y_interpad, y_interpad_err = [], []
    y_full, y_full_err = [], []

    for file in tqdm(os.listdir(folder_name)):
        if file.endswith(".csv"):
            with open(f"{folder_name}/{file}", "r") as f:
                df = pd.read_csv(f)

            voltage = int(file.split("_")[-1][:-5])

            if voltage in exclusions:
                continue

            y_borders, _, _ = plot_y_profile(df, voltage, threshold=0.9)
            x_borders = plot_x_profile(df, voltage, threshold=0.8)

            jitters, jitter_errors = plot_jitter(
                df,
                percentage,
                x_borders=x_borders,
                y_borders=y_borders,
                fit_options={
                    "bins": 100,
                    "left_lim": 95,
                    "right_lim": 98.8,
                    "peak_left": 95,
                    "peak_right": 98.8,
                },
                make_plot=False,
            )

            x.append(voltage)

            if "ch_3" in jitters or "ch_4" in jitters:
                y_ch1.append(1e3 * jitters.get("ch_3", np.nan))
                y_ch1_err.append(1e3 * jitter_errors.get("ch_3", 0))

                y_ch2.append(1e3 * jitters.get("ch_4", np.nan))
                y_ch2_err.append(1e3 * jitter_errors.get("ch_4", 0))
                start_channel = 3
            else:
                y_ch1.append(1e3 * jitters.get("ch_1", np.nan))
                y_ch1_err.append(1e3 * jitter_errors.get("ch_1", 0))

                y_ch2.append(1e3 * jitters.get("ch_2", np.nan))
                y_ch2_err.append(1e3 * jitter_errors.get("ch_2", 0))
                start_channel = 1

            y_interpad.append(1e3 * jitters.get("interpad", np.nan))
            y_interpad_err.append(1e3 * jitter_errors.get("interpad", 0))

            y_full.append(1e3 * jitters.get("full", np.nan))
            y_full_err.append(1e3 * jitter_errors.get("full", 0))

    if len(x) > 0:
        sorted_tuples = sorted(
            zip(
                x,
                y_ch1,
                y_ch1_err,
                y_ch2,
                y_ch2_err,
                y_interpad,
                y_interpad_err,
                y_full,
                y_full_err,
            )
        )
        (
            x_sorted,
            y_ch1_sorted,
            y_ch1_err_sorted,
            y_ch2_sorted,
            y_ch2_err_sorted,
            y_interpad_sorted,
            y_interpad_err_sorted,
            y_full_sorted,
            y_full_err_sorted,
        ) = zip(*sorted_tuples)

        line1 = ax1.errorbar(
            x_sorted,
            y_ch1_sorted,
            yerr=y_ch1_err_sorted,
            marker="o",
            capsize=4,
            label=f"{wafer_type} {sample_name} Ch{start_channel}",
        )
        color = line1[0].get_color()

        ax1.errorbar(
            x_sorted,
            y_ch2_sorted,
            yerr=y_ch2_err_sorted,
            marker="o",
            capsize=4,
            label=f"{wafer_type} {sample_name} Ch{start_channel+1}",
            linestyle="--",
            color=color,
        )

        ax2.errorbar(
            x_sorted,
            y_interpad_sorted,
            yerr=y_interpad_err_sorted,
            marker="o",
            capsize=4,
            label=f"{wafer_type} {sample_name}",
            color=color,
        )

        ax3.errorbar(
            x_sorted,
            y_full_sorted,
            yerr=y_full_err_sorted,
            marker="o",
            capsize=4,
            label=f"{wafer_type} {sample_name}",
            color=color,
        )

ax1.set_title("Channels")
ax1.set_xlabel("Bias voltage, [V]")
ax1.set_ylabel("Jitter, [ps]")
ax1.grid(True)
ax1.legend()

ax2.set_title("Inter-pad")
ax2.set_xlabel("Bias voltage, [V]")
ax2.grid(True)
ax2.legend()

ax3.set_title("Full sample")
ax3.set_xlabel("Bias voltage, [V]")
ax3.grid(True)
ax3.legend()

ax1.set_ylim(0, 120)
ax2.set_ylim(0, 120)
ax3.set_ylim(0, 120)

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

sample_list = [
    ["W11", "TW5_V1", []],
    ["W11", "TW5_V2", []],
    ["W11", "TW1_V2", []],
    ["W11", "TW2_V2", []],
]

for wafer_type, sample_name, exclusions in sample_list:
    folder_name = f"{data_folder}/{wafer_type}/{sample_name}"

    x = []
    y_ch1 = []
    y_ch2 = []
    y_interpad = []
    y_full = []

    for file in tqdm(os.listdir(folder_name)):
        if file.endswith(".csv"):
            with open(f"{folder_name}/{file}", "r") as f:
                df = pd.read_csv(f)

            voltage = int(file.split("_")[-1][:-5])

            if voltage in exclusions:
                continue

            y_borders, _, _ = plot_y_profile(
                df, voltage, threshold=0.9, make_plot=False
            )
            x_borders = plot_x_profile(df, voltage, threshold=0.8, make_plot=False)

            charges = plot_mean_charge_map(
                df, voltage, x_borders=x_borders, y_borders=y_borders, make_plot=False
            )

            x.append(voltage)

            if "ch_3" in charges or "ch_4" in charges:
                y_ch1.append(charges.get("ch_3", np.nan))
                y_ch2.append(charges.get("ch_4", np.nan))
                start_channel = 3
            else:
                y_ch1.append(charges.get("ch_1", np.nan))
                y_ch2.append(charges.get("ch_2", np.nan))
                start_channel = 1

            y_interpad.append(charges.get("interpad", np.nan))
            y_full.append(charges.get("full", np.nan))

    if len(x) > 0:
        sorted_tuples = sorted(zip(x, y_ch1, y_ch2, y_interpad, y_full))
        x_sorted, y_ch1_sorted, y_ch2_sorted, y_interpad_sorted, y_full_sorted = zip(
            *sorted_tuples
        )

        (line1,) = ax1.plot(
            x_sorted,
            y_ch1_sorted,
            marker="o",
            label=f"{wafer_type} {sample_name} Ch{start_channel}",
        )
        color = line1.get_color()
        ax1.plot(
            x_sorted,
            y_ch2_sorted,
            marker="o",
            label=f"{wafer_type} {sample_name} Ch{start_channel+1}",
            linestyle="--",
            color=color,
        )

        ax2.plot(
            x_sorted,
            y_interpad_sorted,
            marker="o",
            label=f"{wafer_type} {sample_name}",
            color=color,
        )

        ax3.plot(
            x_sorted,
            y_full_sorted,
            marker="o",
            label=f"{wafer_type} {sample_name}",
            color=color,
        )

ax1.set_title("Channels")
ax1.set_xlabel("Bias voltage, [V]")
ax1.set_ylabel(r"Mean Charge, [V$\cdot$ns]")
ax1.grid(True)
ax1.legend()

ax2.set_title("Inter-pad")
ax2.set_xlabel("Bias voltage, [V]")
ax2.grid(True)
ax2.legend()

ax3.set_title("Full sample")
ax3.set_xlabel("Bias voltage, [V]")
ax3.grid(True)
ax3.legend()

ax1.set_ylim(0, 3)
ax2.set_ylim(0, 3)
ax3.set_ylim(0, 3)


plt.tight_layout()
plt.show()